# CS444 Final Project: Cross-Modal Alignment in Image Captioning

This notebook is the presentation and experiment runner for the final project. Helper code lives in `src/cs444_captioning/`, while this notebook focuses on running experiments, showing tables, plotting figures, and writing analysis.

Current stage: baseline implementation, `Vanilla ViT + GPT-2`.

## Experiment Matrix

| Experiment | Encoder | Mapper | Decoder | Status |
|---|---|---|---|---|
| `vit_no_mapper` | `google/vit-base-patch16-224-in21k` | No | `gpt2` | Baseline |
| `vit_mlp_mapper` | `google/vit-base-patch16-224-in21k` | Yes | `gpt2` | Next |
| `clip_no_mapper` | `openai/clip-vit-base-patch16` | No | `gpt2` | Next |
| `clip_mlp_mapper` | `openai/clip-vit-base-patch16` | Yes | `gpt2` | Next |

## 1. Setup

In [ ]:
# Optional install cell. Uncomment if needed.
# !pip install -q transformers==4.44.0 datasets evaluate accelerate sacrebleu nltk pycocoevalcap
# !pip install -q pandas matplotlib seaborn tqdm pillow scikit-learn

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

sys.path.append(str(Path.cwd() / "src"))

from cs444_captioning.config import ProjectConfig, ensure_output_root
from cs444_captioning.data import (
    Flickr8kCaptionDataset,
    auto_find_data_root,
    build_flickr8k_vizwiz_splits,
    collate_caption_batch,
    find_caption_file,
    find_image_dir,
    load_flickr8k_captions,
    load_flickr8k_dataframe,
    split_by_image,
)
from cs444_captioning.evaluation import evaluate_predictions
from cs444_captioning.generation import generate_predictions
from cs444_captioning.modeling import build_vit_gpt2_baseline, load_processors, parameter_count
from cs444_captioning.training import train_model
from cs444_captioning.utils import describe_device, set_seed
from cs444_captioning.visualization import (
    plot_metric_bars,
    plot_training_history,
    show_dataset_example,
    show_predictions,
)

CFG = ProjectConfig()
set_seed(CFG.seed)
DEVICE = describe_device()
ensure_output_root(CFG)
CFG.to_display_dict()

## 2. Dataset

Supported Flickr8k layouts:

- Kaggle `adityajn105/flickr8k`: `Images/` plus `captions.txt`.
- Older format: `Flicker8k_Dataset/` plus `Flickr8k.token.txt`.

Optional VizWiz layout:

- `data/vizwiz/annotations/train.json` plus `data/vizwiz/train/`.
- `data/vizwiz/annotations/val.json` or `validation.json` plus `data/vizwiz/val/` or `validation/`.

In [ ]:
# Optional Kaggle download. Use this only if Kaggle credentials are configured.
# !pip install -q kaggle
# !mkdir -p data/flickr8k
# !kaggle datasets download -d adityajn105/flickr8k -p data/flickr8k --unzip
# CFG.data_root = Path("./data/flickr8k")

In [ ]:
USE_VIZWIZ = False
VIZWIZ_ROOT = Path("./data/vizwiz")
VIZWIZ_TRAIN_IMAGES = 10000
VIZWIZ_VAL_IMAGES = 1000

if USE_VIZWIZ:
    train_df, val_df, test_df, data_meta = build_flickr8k_vizwiz_splits(
        flickr8k_root=CFG.data_root,
        vizwiz_root=VIZWIZ_ROOT,
        vizwiz_train_images=VIZWIZ_TRAIN_IMAGES,
        vizwiz_val_images=VIZWIZ_VAL_IMAGES,
        seed=CFG.seed,
    )
    image_dir = Path(".")
    captions_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print("Dataset mode: Flickr8k + VizWiz")
    print(data_meta)
else:
    CFG.data_root = auto_find_data_root(CFG.data_root)
    captions_df, image_dir, caption_file = load_flickr8k_dataframe(CFG.data_root)
    train_df, val_df, test_df = split_by_image(captions_df, seed=CFG.seed)
    print("Dataset mode: Flickr8k only")
    print("Data root:", CFG.data_root)
    print("Caption file:", caption_file)
    print("Image dir:", image_dir)

summary_df = pd.DataFrame([
    {"split": "train", "caption_rows": len(train_df), "unique_images": train_df["image"].nunique(), "sources": ", ".join(sorted(train_df.get("source", pd.Series(["unknown"])).dropna().unique()))},
    {"split": "val", "caption_rows": len(val_df), "unique_images": val_df["image"].nunique(), "sources": ", ".join(sorted(val_df.get("source", pd.Series(["unknown"])).dropna().unique()))},
    {"split": "test", "caption_rows": len(test_df), "unique_images": test_df["image"].nunique(), "sources": ", ".join(sorted(test_df.get("source", pd.Series(["unknown"])).dropna().unique()))},
])

display(summary_df)
display(captions_df.head())

In [ ]:
image_processor, tokenizer = load_processors(CFG)

train_dataset = Flickr8kCaptionDataset(train_df, image_dir, image_processor, tokenizer, CFG.max_length)
val_dataset = Flickr8kCaptionDataset(val_df, image_dir, image_processor, tokenizer, CFG.max_length)
test_dataset = Flickr8kCaptionDataset(test_df, image_dir, image_processor, tokenizer, CFG.max_length)

show_dataset_example(train_df, image_dir)

## 3. Baseline: Vanilla ViT + GPT-2

Architecture:

- Encoder: `google/vit-base-patch16-224-in21k`
- Decoder: `gpt2`
- Bridge: GPT-2 cross-attention
- Encoder frozen

In [ ]:
baseline_model = build_vit_gpt2_baseline(CFG, tokenizer, DEVICE, freeze_encoder=True)
total_params, trainable_params = parameter_count(baseline_model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_caption_batch,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.eval_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_caption_batch,
)

batch = next(iter(train_loader))
baseline_model.eval()
with torch.no_grad():
    out = baseline_model(
        pixel_values=batch["pixel_values"].to(DEVICE),
        labels=batch["labels"].to(DEVICE),
    )
print("Sanity check loss:", float(out.loss.detach().cpu()))
print("Logits shape:", tuple(out.logits.shape))

## 4. Debug Baseline Run

Run this first. It trains on a tiny subset to verify the full pipeline before using GPU time on the full dataset.

In [ ]:
RUN_DEBUG_BASELINE = True

if RUN_DEBUG_BASELINE:
    debug_cfg = ProjectConfig(
        data_root=CFG.data_root,
        output_root=CFG.output_root,
        num_epochs=1,
        batch_size=4,
        eval_batch_size=4,
        num_workers=0,
        max_length=CFG.max_length,
    )
    debug_train_df = train_df.sample(min(len(train_df), debug_cfg.debug_train_samples), random_state=CFG.seed).reset_index(drop=True)
    debug_val_df = val_df.sample(min(len(val_df), debug_cfg.debug_val_samples), random_state=CFG.seed).reset_index(drop=True)

    debug_train_dataset = Flickr8kCaptionDataset(debug_train_df, image_dir, image_processor, tokenizer, debug_cfg.max_length)
    debug_val_dataset = Flickr8kCaptionDataset(debug_val_df, image_dir, image_processor, tokenizer, debug_cfg.max_length)
    debug_train_loader = DataLoader(debug_train_dataset, batch_size=debug_cfg.batch_size, shuffle=True, num_workers=0, collate_fn=collate_caption_batch)
    debug_val_loader = DataLoader(debug_val_dataset, batch_size=debug_cfg.eval_batch_size, shuffle=False, num_workers=0, collate_fn=collate_caption_batch)

    debug_model = build_vit_gpt2_baseline(debug_cfg, tokenizer, DEVICE, freeze_encoder=True)
    debug_history = train_model(debug_model, debug_train_loader, debug_val_loader, tokenizer, DEVICE, debug_cfg, "debug_vit_no_mapper")
    display(debug_history)
    plot_training_history(debug_history, "Debug Baseline Training")

In [ ]:
if RUN_DEBUG_BASELINE:
    debug_predictions = generate_predictions(
        debug_model,
        test_df,
        image_dir,
        image_processor,
        tokenizer,
        debug_cfg,
        DEVICE,
        max_images=8,
        experiment_name="debug_vit_no_mapper",
    )
    show_predictions(debug_predictions, image_dir, n=3)

    debug_metrics = evaluate_predictions(debug_predictions)
    debug_results_df = pd.DataFrame([
        {"Experiment": "debug_vit_no_mapper", "Encoder": "ViT", "Mapper": "No", **debug_metrics}
    ])
    display(debug_results_df)
    plot_metric_bars(debug_results_df)

## 5. Full Baseline Run

After the debug run works, change `RUN_FULL_BASELINE` to `True`. This is the first official result row for the final report.

In [ ]:
RUN_FULL_BASELINE = False

if RUN_FULL_BASELINE:
    CFG.num_epochs = 10
    CFG.batch_size = 8
    CFG.eval_batch_size = 8
    CFG.learning_rate = 5e-5

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, collate_fn=collate_caption_batch)
    val_loader = DataLoader(val_dataset, batch_size=CFG.eval_batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate_caption_batch)

    full_baseline_model = build_vit_gpt2_baseline(CFG, tokenizer, DEVICE, freeze_encoder=True)
    baseline_history = train_model(full_baseline_model, train_loader, val_loader, tokenizer, DEVICE, CFG, "vit_no_mapper")
    display(baseline_history)
    plot_training_history(baseline_history, "Full Baseline Training")

    baseline_predictions = generate_predictions(
        full_baseline_model,
        test_df,
        image_dir,
        image_processor,
        tokenizer,
        CFG,
        DEVICE,
        max_images=None,
        experiment_name="vit_no_mapper",
    )
    show_predictions(baseline_predictions, image_dir, n=5)

    baseline_metrics = evaluate_predictions(baseline_predictions)
    baseline_results_df = pd.DataFrame([
        {"Experiment": "vit_no_mapper", "Encoder": "ViT", "Mapper": "No", **baseline_metrics}
    ])
    display(baseline_results_df)
    plot_metric_bars(baseline_results_df)

## 6. Final Results Table

After all four experiments are implemented, append each result row here and use this table directly in the report.

In [ ]:
final_results = []

if "baseline_results_df" in globals():
    final_results.extend(baseline_results_df.to_dict("records"))
elif "debug_results_df" in globals():
    final_results.extend(debug_results_df.to_dict("records"))

results_df = pd.DataFrame(final_results)
display(results_df)
plot_metric_bars(results_df)

## 7. Analysis Notes

Fill this section after running all four experiments.

Questions to answer:

1. Does CLIP pre-alignment improve BLEU-4, METEOR, or CIDEr compared with vanilla ViT?
2. Does the MLP mapper improve alignment for vanilla ViT?
3. Does CLIP + MLP combine well, or does one component dominate?
4. Which examples fail, and what do those failures reveal about cross-modal alignment?

## 8. Next Implementation Targets

1. Add a custom model wrapper with optional `MLPMapper`.
2. Add `openai/clip-vit-base-patch16` as the second encoder.
3. Replace the baseline-specific experiment code with a unified experiment loop.
4. Add qualitative comparison: same image, four model predictions, references.
5. Add attention visualization if time allows.